# 6 — Investment

Companion to **section 7**. Capacity becomes a *variable*, charged to the
objective at an annualised cost. The Danish zones are rebuilt from scratch
from a menu of candidates, the neighbours keep their fleets and may add to
them, and every zone's demand is at the 2050 horizon.

Runtime: two solves of about a minute each at 168 segments.
The note's own figures use 1,095.

> **One file you have to build first.** This section prices its candidates
> from a technology-cost table that is *not* shipped, because its source's
> licence does not permit redistribution. The cell below tells you what to
> run if it is missing; it is one small download at a pinned version.

In [ ]:
# Make the note's models importable, wherever you launched Jupyter from.
import sys
from pathlib import Path

here = Path.cwd()
note = next(p for p in [here, *here.parents] if (p / "model" / "dispatch.py").exists())
sys.path.insert(0, str(note))
sys.path.insert(0, str(note / "pipeline"))   # the run scripts' helpers
PROCESSED = note / "data" / "processed"

import pandas as pd
import matplotlib.pyplot as plt

print(f"note root: {note}")

In [ ]:
# The investment models read their cost tables when imported, so a missing
# table shows up here rather than at the first solve.
try:
    from model import greenfield
except FileNotFoundError as missing:
    raise SystemExit(
        f"Missing {Path(missing.filename).name}. This file is not shipped with the "
        "repository: it is our reshaped subset of an external technology-cost "
        "database whose compiled outputs carry no single stated licence, so you "
        "build it yourself, once, with\n\n    python data/prepare.py --costs-only\n\n"
        "from the note's directory (one small download per vintage, at a pinned "
        "version). Notebooks 00-05 and 09 run without it."
    ) from None

costs = pd.read_csv(PROCESSED / f"technology_costs_full_{greenfield.FORWARD_HORIZON}.csv",
                    index_col="technology")
print(f"cost table: {len(costs)} technologies, {greenfield.FORWARD_HORIZON} vintage")

## The screening curve

Before solving anything: what does each candidate cost to *have* (EUR per
MW-year, the annuity of eq. investment:annuity plus fixed O&M) and to *run*
(EUR/MWh)? These two numbers per technology are the whole screening curve of
figure 7.1, and they decide who gets built at which utilisation.

In [ ]:
cc = greenfield.candidate_costs(costs)
cc["fixed_keur_per_mw_yr"] = cc["fixed_eur_mw_yr"] / 1e3
cc[["fixed_keur_per_mw_yr", "marginal_cost", "emission_rate"]].round(2)

## What should Denmark build?

The reference case: every zone pays the ETS price on its emissions, no
budget. The solve returns the optimal Danish capacity and, for each built
technology, the full-load hours it actually achieves — read those against
the screening curve.

In [ ]:
from run_greenfield import year_weather, REFERENCE_WEATHER_YEAR

HOURS = 168
NC = PROCESSED / "network_eur_bz_2024.nc"
weather = year_weather(REFERENCE_WEATHER_YEAR)   # all twelve zones on one ERA5 year

n = greenfield.build(NC, costs, hours=HOURS, weather=weather)
greenfield.solve(n)

caps = greenfield.dk_capacity(n)
w = n.snapshot_weightings.objective
dk = n.generators[n.generators.bus.isin(greenfield.DK_ZONES) & n.generators.p_nom_extendable]
energy = n.generators_t.p[dk.index].mul(w, axis=0).sum().groupby(dk.carrier).sum()
flh = (energy / dk.groupby("carrier")["p_nom_opt"].sum()).replace([float("inf")], 0)
out = pd.DataFrame({"capacity (MW)": caps, "full-load hours": flh}).fillna(0)
print(f"Danish emissions: {greenfield.dk_emissions(n)/1e6:.2f} Mt/yr")
out.round(0)

Section 7.2 of the note: every built plant recovers exactly its annualised
cost from the scarcity rents it earns — the zero-profit condition — and
anything that would not is not built. A candidate with zero capacity is one
whose screening line lies above the others at every utilisation it could
reach.

## A Danish CO2 budget

Now replace the price with a *quantity*: a territorial cap on Danish
emissions at 10% of the reference case. The dual of the budget is the carbon
price Denmark would need to hit it — and, because the neighbours are
unconstrained, some of the abatement is imports.

In [ ]:
budget = 0.10 * greenfield.dk_emissions(n)
n_cap = greenfield.build(NC, costs, hours=HOURS, weather=weather, co2_budget=budget)
sigma = greenfield.solve(n_cap)
print(f"shadow price of the budget: {sigma:.0f} EUR/t")

pd.DataFrame({
    "reference (MW)": greenfield.dk_capacity(n),
    "with budget (MW)": greenfield.dk_capacity(n_cap),
}).fillna(0).round(0)

## Your turn

1. `autarky=True` cuts Denmark off. Trade turns out to be load-bearing: what
   gets built instead, and what happens to the shadow price of the budget?
2. Change the discount rate (`r=0.03`) and reconcile the new mix with the
   annuity formula in Appendix B of the note. Which technologies gain?
3. Sweep the common carbon price instead of a Danish budget:
   `ets_price=tau` for a few values. That is figure 7.6, and the reason the
   note says a Danish budget and a European price are different questions.
4. `greenfield.shed_energy(n)` reports how much load the model chose not to
   serve. Why is it not zero, and at what price does it happen?

In [ ]:
# Try it here.